# Tutorial Datapizza AI: OpenAI Client & Chatbot 🍕

Questo tutorial esplora l'utilizzo della libreria **Datapizza AI** concentrandosi sul **Client OpenAI** e le sue funzionalità avanzate (Streaming, JSON Mode, Vision), come mostrato nella documentazione ufficiale.

Alla fine del notebook, implementeremo un **Chatbot completo** utilizzando le primitive di memoria del framework.

### Indice
1. **Setup**: Installazione e Configurazione.
2. **Chat API Base**: Completamento testo standard.
3. **Streaming**: Ricevere token in tempo reale.
4. **JSON Mode**: Ottenere output strutturati.
5. **GPT Vision**: Analisi di immagini.
6. **Implementazione Chatbot**: Costruzione di un loop conversazionale con memoria.

## 1. Setup e Installazione

Installiamo il core di Datapizza e il client specifico per OpenAI.

In [ ]:
# Installazione dei pacchetti necessari
!pip install datapizza-ai datapizza-ai-clients-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.6/182.6 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.0/98.0 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 328.6/328.6 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 243.4/243.4 kB 15.5 MB/s eta 0:00:00
  Attempting uninstall: rich
    Found existing installation: rich 13.9.4
    Uninstalling rich-13.9.4:
      Successfully uninstalled rich-13.9.4
  Attempting uninstall: openai
    Found existing installation: openai 1.109.1
    Uninstalling openai-1.109.1:
      Successfully uninstalled openai-1.109.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.29.1 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.


In [ ]:
import os
import getpass

# Configurazione API Key
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Inserisci la tua OpenAI API Key: ")

Inserisci la tua OpenAI API Key: ··········


## 2. Inizializzazione Client e Chat API

Il modulo `datapizza.clients.openai` fornisce un wrapper robusto attorno alle API di OpenAI. Inizializziamo il client.

In [ ]:
from datapizza.clients.openai import OpenAIClient
from datapizza.memory import Memory
from datapizza.type import ROLE, TextBlock

memory=Memory()

# Inizializzazione del client standard
# model: puoi usare "gpt-4o", "gpt-4-turbo", etc.
client = OpenAIClient(
    model="gpt-5.1",
    temperature=0.7,
    api_key=os.environ["OPENAI_API_KEY"],
    system_prompt="Sei un esperto di pizza italiana.",

)

# Test semplice di connessione (Chat API)
message = "Qual è il segreto per un buon impasto?"
# Second interaction
response = client.invoke(message)
memory.add_turn(TextBlock(content=message), role=ROLE.USER)
memory.add_turn(response.content, role=ROLE.ASSISTANT)
print(response.text)

# Second interaction - the model remembers Alice
response2 = client.invoke("Mi ripeti quello che hai detto ma in rima?", memory=memory)
print(response2.text)  # Should mention Alice



Il “segreto” non è uno solo, ma l’equilibrio di alcuni fattori fondamentali:

1. **Farina giusta**
   - Per pizza italiana classica: farina 0 o 00 con W tra 260 e 320 (media/alta forza).
   - Una farina troppo debole non regge lunghe lievitazioni, una troppo forte richiede tempi lunghi e buona gestione.

2. **Idratazione corretta**
   - Per iniziare: 60–65% (600–650 g acqua per 1 kg di farina).
   - Più acqua = impasto più leggero e alveolato, ma più difficile da gestire.
   - Aumenta l’idratazione solo quando hai mano sull’impasto.

3. **Poca lievito, tanto tempo**
   - Meglio poco lievito e lunga lievitazione/maturazione in frigo.
   - Esempio casalingo: 1–2 g di lievito di birra fresco per 1 kg di farina, 18–24 ore di maturazione in frigo (4–6 °C).
   - Il tempo permette sviluppo di aromi e digeribilità.

4. **Sale e tempi**
   - Sale: 2–2,5% sul peso della farina (20–25 g per 1 kg).
   - Non metterlo a diretto contatto con il lievito all’inizio: prima acqua + farina + lievito, poi 

## 3. Dati strutturati

Per integrare l'AI nel software, spesso serve un JSON invece di testo libero. OpenAI (e Datapizza-AI) supportano le risposte strutturate.

In [ ]:
from pydantic import BaseModel

class Person(BaseModel):
    name: str
    age: int
    occupation: str

response = client.structured_response(
    input="Create a profile for a software engineer",
    output_cls=Person
)

person = response.structured_data[0]
print(f"Name: {person.name}")
print(f"Age: {person.age}")
print(f"Occupation: {person.occupation}")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: your-api-key. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

## 5. GPT Vision (Modelli Multimodali)

Sezione dedicata all'analisi di immagini tramite modelli come `gpt-4o`.

In [ ]:
# URL di un'immagine di esempio (una pizza!)
image_url = "https://upload.wikimedia.org/wikipedia/commons/a/a3/Eq_it-na_pizza-margherita_sep2005_sml.jpg"

# Costruiamo il messaggio multimodale
vision_messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Descrivi cosa vedi in questa immagine."}, # Testo
            {"type": "image_url", "image_url": {"url": image_url}}          # Immagine
        ]
    }
]

print("Analisi immagine in corso...")
vision_response = client.invoke(messages=vision_messages)
print(f"Descrizione AI: {vision_response.content}")

## 6. Implementazione Chatbot con Memoria

Ora uniamo i pezzi per creare un semplice chatbot che si "ricorda" della conversazione. Useremo la classe `Memory` di Datapizza per gestire la cronologia.

In [ ]:
from datapizza.memory import Memory
from datapizza.types import TextBlock, ROLE

def run_chatbot():
    print("🍕 Datapizza Chatbot (scrivi 'esci' per terminare) 🍕")

    # 1. Inizializziamo la memoria
    memory = Memory()

    # 2. Aggiungiamo il system prompt alla memoria
    memory.add_turn(
        TextBlock(content="Sei un assistente amichevole e divertente."),
        role=ROLE.SYSTEM
    )

    while True:
        # Input utente
        user_input = input("Tu: ")
        if user_input.lower() in ["esci", "quit", "exit"]:
            print("Chat terminata. Ciao!")
            break

        # 3. Aggiungiamo l'input utente alla memoria
        memory.add_turn(
            TextBlock(content=user_input),
            role=ROLE.USER
        )

        # 4. Chiamata al client passando TUTTA la storia (recuperata da memory)
        # Usa .invoke() per inviare la history
        response = client.invoke(messages=memory.get_history())

        bot_reply = response.content
        print(f"Bot: {bot_reply}")

        # 5. Aggiungiamo la risposta del bot alla memoria
        memory.add_turn(
            TextBlock(content=bot_reply),
            role=ROLE.ASSISTANT
        )

# Decommenta per eseguire il chatbot interattivo
run_chatbot()